In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [2]:
from src.dataset import ImageDataset
from torch.utils.data import DataLoader

annotations_file_trainval = Path("../data/preprocessed/trainval/annotations.csv")
img_dir_trainval = Path("../data/preprocessed/trainval/Images")

trainval_dataset = ImageDataset(annotations_file_trainval, img_dir_trainval)
trainval_dl = DataLoader(trainval_dataset, batch_size=32, shuffle=True)

In [3]:
from src.model import Model

model = Model()

In [26]:
from src.configs import S, B, C, IDX_TO_CLASS
from src.utils import convert_xywh_coords

def decode_preds(preds_batch):
    decoded_preds = []

    for pred in preds_batch:
        pred = pred.reshape((S, S, C + B * 5))
        objects = []

        for i in range(S):
            for j in range(S):
                pred_cell = pred[i][j]

                pred_class_idx = pred_cell[:20].argmax().item()
                pred_class_prob = pred_cell[pred_class_idx].item()
                pred_class = IDX_TO_CLASS[pred_class_idx]
                
                pred_1_confidence = (pred_cell[24].item() * \
                                     pred_class_prob,)
                pred_2_confidence = (pred_cell[29].item() * \
                                     pred_class_prob,)

                bbox_1 = convert_xywh_coords(pred_cell[20:24], i, j, False, True)
                bbox_2 = convert_xywh_coords(pred_cell[25:29], i, j, False, True)
                
                objects.append((pred_class,) + pred_1_confidence + bbox_1)
                objects.append((pred_class,) + pred_2_confidence + bbox_2)

        decoded_preds.append(objects)

    return decoded_preds

In [5]:
X_batch, y_batch = next(iter(trainval_dl))

X_batch.shape, y_batch.shape

(torch.Size([32, 3, 224, 224]), torch.Size([32, 7, 7, 30]))

In [6]:
preds = model(X_batch)
preds.shape

torch.Size([32, 1470])

In [7]:
preds = preds.reshape((preds.shape[0], S, S, B * 5 + C))
preds.shape

torch.Size([32, 7, 7, 30])

In [28]:
decoded_preds = decode_preds(preds)
len(decoded_preds), len(decoded_preds[0])

(32, 98)

In [39]:
CONFIDENCE_THRESHOLD = 0.375

def confidence_threshold(decoded_preds):
    valid_preds = []

    for image in decoded_preds:
        image_preds = []
        for pred in image:
            if pred[1] > CONFIDENCE_THRESHOLD:
                image_preds.append(pred)

        if image_preds:
            valid_preds.append(image_preds)
            
    return valid_preds

In [41]:
valid_preds = confidence_threshold(decoded_preds)
valid_preds

[[('sofa',
   0.4509951529202958,
   -4.606571197509766,
   53.986663818359375,
   78.99382019042969,
   62.076507568359375),
  ('bicycle',
   0.7086426417941425,
   171.10858154296875,
   82.01881408691406,
   219.548828125,
   88.16122436523438),
  ('boat',
   0.3796610147494661,
   183.93832397460938,
   143.39627075195312,
   218.3892822265625,
   102.33279418945312)],
 [('chair',
   0.39342718000723664,
   30.914796829223633,
   -26.443403244018555,
   87.39605712890625,
   29.198713302612305)],
 [('sofa',
   0.40431039231781085,
   235.3980712890625,
   -35.76116943359375,
   81.13399505615234,
   13.626947402954102),
  ('aeroplane',
   0.5109307299902568,
   195.76751708984375,
   6.537708759307861,
   214.3408203125,
   8.254890441894531),
  ('horse',
   0.5243837048932392,
   -4.714864730834961,
   -68.97616577148438,
   50.099647521972656,
   138.41226196289062),
  ('sofa',
   0.5457564215351596,
   91.9547348022461,
   24.822052001953125,
   23.76519775390625,
   34.68055725